Extract Meteo bulletins and compile into parquet files, one for each year.

Extract Thermo zip files and compile into parquet files, one for each year.

Extract NOAA CPD2 tarballs and compile into parquet files, one for each year.

Extract AE33 files and compile into parquet files, one for each month.

Extract G2401 tarballs and compile into parquet files, one for each year.

NB: Run this *after* having organized the files using mch_incoming.ipynb.

joerg.klausen@meteoswiss.ch

In [ ]:
import os
import yaml
import housekeeping.organize_files as hk
import monitoring.file_coverage as fc
from processing.meteo import Meteo
from processing.thermo import Thermo
from processing.cpd2 import CPD2
from processing.ae33 import AE33
from processing.g2401 import G2401

with open("mch-config.yml", "r") as fh:
    cfg = yaml.safe_load(fh)
    fh.close()

# organize newly arrived files on MeteoSwiss fileshare
mkn = cfg["mkn"]
n = hk.organize_files(mkn, branch="incoming", verbosity=1)

# show and plot file statistics
days = 7
stats = fc.get_file_coverage(cfg=mkn, days=days)
display(df:=fc.print_coverage(stats=stats, days=days))
fc.plot_coverage(stats=stats, days=days)

# root path to folders at MeteoSwiss
root = cfg["mkn"]["root"]

# root paths to folders
source = os.path.join(root, "incoming")
archive = os.path.join(root, "archive")
target = os.path.join("data", "level1")
issues = os.path.join(root, "incoming_with_issues")
logs = os.path.join("logging")

# remove empty directories under source
# os.system(f"find {source} -empty -type d)
# NB: no questions asked!
# os.system(f"find {source} -empty -type d -delete")

# months, years to cover
months = ["{:02d}".format(mm) for mm in range(1, 13, 1)]
# years = ["2020", "2021", "2022", "2023", "2024"]
years = ["2024"]

# instantiate instruments (rather, data types)
met = Meteo(log=os.path.join(logs, "meteo.log"))
thermo = Thermo(log=os.path.join(logs, "thermo.log"))
cpd2 = CPD2(log=os.path.join(logs, "cpd2.log"))
ae33 = AE33(log=os.path.join(logs, "ae33.log"))
g2401 = G2401(log=os.path.join(logs, "g2401.log"))

In [ ]:
# process all instruments with yearly level1 files
for year in years:
    if os.path.exists(source):
        met.compile_vrxa00_to_parquet(
            source=os.path.join(source, "meteo"), 
            target=os.path.join(target, "meteo"), 
            year=year, 
            archive=os.path.join(archive, "meteo"), 
            issues=os.path.join(issues, "meteo"),
            )
        thermo.compile_thermo_to_parquet(
            source=os.path.join(source, "tei49c"),
            target=os.path.join(target, "tei49c"),
            base=year, 
            archive=os.path.join(archive, "tei49c"),
            issues=os.path.join(issues, "tei49c"),
            )        
        thermo.compile_thermo_to_parquet(
            source=os.path.join(source, "tei49i"),
            target=os.path.join(target, "tei49i"),
            base=year, 
            archive=os.path.join(archive, "tei49i"),
            issues=os.path.join(issues, "tei49i"),
            )
        # cpd2.tarballs_to_parquet(
        #     source=os.path.join(source, "aerosol", year), 
        #     target=os.path.join(target, "aerosol", year), 
        #     archive=os.path.join(archive, "aerosol", year), 
        #     issues=os.path.join(issues, "aerosol"),
        #     )

# process all instruments with monthly level1 files
for year in years:
    for month in months:
        if os.path.exists(source):
            ae33.zipfiles_to_parquet(
                source=os.path.join(source, "ae33", "data", year, month), 
                target=os.path.join(target, year, month), 
                archive=os.path.join(archive, "ae33", "data", year, month), 
                issues=os.path.join(issues, "ae33"), 
                plot=True,
                )
            g2401.compile_g2401_to_parquet(
                source=os.path.join(source, "g2401"), 
                target=os.path.join(target, "g2401"), 
                year=year, 
                archive=os.path.join(archive, "g2401"), 
                issues=os.path.join(issues, "g2401"),
                )